# 球员评估测试 - 2022世界杯决赛

本notebook用于测试球员评估功能在2022世界杯决赛数据上的运行。

**比赛**: 阿根廷 vs 法国 (Game ID: 10517)

**测试内容**:
1. 防守球员影响力分析
2. 防守球员移除实验
3. 球员表现评估
4. 数据导出和可视化

## 1. 导入必要的库

In [1]:
import os
import pickle
import sys
import pandas as pd
import numpy as np
import random
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader

# 添加父目录到Python路径（用于导入项目模块）
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
    print(f"✓ 已添加父目录到Python路径: {parent_dir}")

import convert_tracking as ct
import create_graph as cg
import scale_graph as sg
import GNNs.convert_data as cd
from GNNs.GAT import GATReceptionPredictor

print("✓ 所有库导入成功")

✓ 已添加父目录到Python路径: e:\JerryWu\Master\SoccerAnalytics\TrackingData_literature_code\Evaluating Defensive Influence Using GATs-main (modified for 2022 WC)
✓ 所有库导入成功


## 2. 配置参数

In [2]:
GAME_ID = '10517'
DATA_DIR = f'Data/{GAME_ID}'
XT_GRID_PATH = 'xT_grid.csv'
MODEL_PATH = f'results/model_final_{GAME_ID}.pth'
SCALER_PATH = f'results/scaler_final_{GAME_ID}.pkl'
SCALED_GRAPHS_PATH = f'results/scaled_graphs_final_{GAME_ID}.pkl'
MAX_GRAPHS = 50
OUTPUT_DIR = f'results/player_eval_{GAME_ID}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"配置完成: 比赛ID={GAME_ID}, 最大图数={MAX_GRAPHS}")

配置完成: 比赛ID=10517, 最大图数=50


## 3. 加载数据

In [3]:
xT_grid = pd.read_csv(XT_GRID_PATH, header=None) if os.path.exists(XT_GRID_PATH) else pd.DataFrame(np.linspace(0, 1, 96).reshape(12, 8))
metadata = ct.get_metadata(GAME_ID)
home_team_name, away_team_name = metadata[2], metadata[3]
rosters_home, rosters_away = metadata[5], metadata[6]

team_rosters_data = []
for player in rosters_home:
    team_rosters_data.append({'game_id': int(GAME_ID), 'player_nickname': player, 'team_name': home_team_name})
for player in rosters_away:
    team_rosters_data.append({'game_id': int(GAME_ID), 'player_nickname': player, 'team_name': away_team_name})
team_rosters = pd.DataFrame(team_rosters_data)

balls_df = pd.read_csv(f'{DATA_DIR}/balls_{GAME_ID}.csv')
events_df = pd.read_csv(f'{DATA_DIR}/events_{GAME_ID}.csv')
players_df = pd.read_csv(f'{DATA_DIR}/players_{GAME_ID}.csv')

print(f"✓ 数据加载完成: {home_team_name} vs {away_team_name}")

✓ 数据加载完成: Argentina vs France


## 4. 加载图和模型

In [4]:
if os.path.exists(SCALED_GRAPHS_PATH):
    with open(SCALED_GRAPHS_PATH, 'rb') as f:
        scaled_graphs = pickle.load(f)[:MAX_GRAPHS]
    print(f"✓ 加载了 {len(scaled_graphs)} 个预缩放的图")
else:
    print("创建新图...")
    graphs = []
    for frameNum in events_df['frameNum'].values[:MAX_GRAPHS]:
        G = cg.create_normalized_graph_directed(players_df, balls_df, events_df, frameNum, home_team_name)
        if G: graphs.append(G)
    graph_scaler = sg.GraphFeatureScaler()
    graph_scaler.fit(graphs)
    scaled_graphs = [graph_scaler.transform_graph(G) for G in graphs]
    print(f"✓ 创建了 {len(scaled_graphs)} 个图")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model = GATReceptionPredictor(15, 6, 32, 16, 16).to(device)
if os.path.exists(MODEL_PATH):
    loaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
loaded_model.eval()

if os.path.exists(SCALER_PATH):
    with open(SCALER_PATH, 'rb') as f:
        loaded_scaler = pickle.load(f)
else:
    from sklearn.preprocessing import MinMaxScaler
    loaded_scaler = MinMaxScaler()
    loaded_scaler.fit([[0, 0], [105, 68]])

print(f"✓ 模型和缩放器加载完成")

创建新图...
✓ 创建了 43 个图
✓ 模型和缩放器加载完成


## 5. 定义分析函数

In [5]:
def get_pitch_value(data, x, y, pitch_length=105, pitch_width=68):
    x = max(1, min(x, 104))
    y = max(1, min(y, 67))
    rows, cols = data.shape[0], data.shape[1]
    col_index = min(int(x / (pitch_length / cols)), cols - 1)
    row_index = min(int(y / (pitch_width / rows)), rows - 1)
    return data.iloc[row_index, col_index]

def calculate_defender_influence(scaled_graph, model, defender, attacker_names, all_names, scaler, xT_grid):
    data = cd.convert_graph_to_pytorch_geometric_reception(scaled_graph)
    model.eval()
    graph_names = [s[0] for s in scaled_graph.nodes(data=True)]
    defender_data = scaled_graph.nodes[defender]
    
    # 处理不同类型的scaler
    if hasattr(scaler, 'position_scaler'):
        defender_x, defender_y = scaler.position_scaler.inverse_transform([[defender_data['features'][0], defender_data['features'][1]]])[0]
    else:
        defender_x, defender_y = scaler.inverse_transform([[defender_data['features'][0], defender_data['features'][1]]])[0]
    
    with torch.no_grad():
        probs, attention_weights = model(data.x, data.edge_index, data.edge_attr, None)
    
    attention_weights_1, attention_weights_2 = attention_weights
    edge_index_1, attn_weights_1 = attention_weights_1
    edge_index_2, attn_weights_2 = attention_weights_2
    
    results = {'distances': [], 'attentions': [], 'influences': [], 'rel_influences': [], 'threats': []}
    total_performance = 0
    
    for attacker in attacker_names:
        attacker_data = scaled_graph.nodes[attacker]
        
        # 处理不同类型的scaler
        if hasattr(scaler, 'position_scaler'):
            x, y = scaler.position_scaler.inverse_transform([[attacker_data['features'][0], attacker_data['features'][1]]])[0]
        else:
            x, y = scaler.inverse_transform([[attacker_data['features'][0], attacker_data['features'][1]]])[0]
        
        distance = np.sqrt((defender_x - x)**2 + (defender_y - y)**2)
        xt_value = get_pitch_value(xT_grid, x, y)
        
        mask_node = graph_names.index(defender)
        target_node = graph_names.index(attacker)
        target_mask = (edge_index_1[1] == target_node)
        source_mask = (edge_index_1[0] == mask_node)
        modify_mask = target_mask & source_mask
        
        if modify_mask.any():
            player_attn1 = attn_weights_1[modify_mask].mean()
            player_attn2 = attn_weights_2[modify_mask].mean()
            defender_attention = float((player_attn1 + player_attn2) / 2)
        else:
            defender_attention = 0.0
        
        with torch.no_grad():
            probs_v2, _ = model(data.x, data.edge_index, data.edge_attr, None, test_mode=True, mask_node_name=defender, target_node_name=attacker, graph=scaled_graph)
        
        attacker_index = all_names.index(attacker)
        prob_before = float(probs[attacker_index])
        prob_after = float(probs_v2[attacker_index])
        influence = prob_after - prob_before
        rel_influence = 100 * (abs(influence) / prob_after) if prob_after > 0 else 0.0
        
        results['distances'].append(distance)
        results['attentions'].append(defender_attention)
        results['influences'].append(influence)
        results['rel_influences'].append(rel_influence)
        results['threats'].append(xt_value)
        total_performance += influence * xt_value * 100
    
    return results, total_performance

print("✓ 分析函数定义完成")

✓ 分析函数定义完成


## 6. 单图测试

In [6]:
example_idx = min(10, len(scaled_graphs) - 1)
graph_example = scaled_graphs[example_idx]
all_names = [s[0] for s in graph_example.nodes(data=True)]
attacker_names = [s[0] for s in graph_example.nodes(data=True) if (s[1]['features'][7] == 1) & (s[0] != 'ball')]
defender_names = [s[0] for s in graph_example.nodes(data=True) if (s[1]['features'][7] == 0) & (s[0] != 'ball')]

print(f"图 {example_idx}: {len(attacker_names)} 进攻球员, {len(defender_names)} 防守球员")

if defender_names:
    test_defender = defender_names[0]
    results, performance = calculate_defender_influence(graph_example, loaded_model, test_defender, attacker_names, all_names, loaded_scaler, xT_grid)
    result_df = pd.DataFrame({'Attacker': attacker_names, 'Distance': results['distances'], 'Attention': results['attentions'], 'Influence': results['influences']})
    print(f"\n防守球员 {test_defender} 的影响力分析:")
    print(result_df.head())
    print(f"总体表现值: {performance:.4f}")

图 10: 11 进攻球员, 11 防守球员

防守球员 Adrien Rabiot 的影响力分析:
              Attacker   Distance  Attention  Influence
0  Alexis Mac Allister  23.536158   0.043453   0.004011
1      Cristian Romero  22.589538   0.043634   0.004654
2    Emiliano Martínez  33.220576   0.043808   0.003495
3       Enzo Fernandez  14.541884   0.043742   0.004052
4       Julian Alvarez  15.197347   0.044792   0.005049
总体表现值: 2.8658


## 7. 批量分析

In [7]:
print(f"开始批量分析 {len(scaled_graphs)} 个图...\n")
all_defender_dataframes = []

for i, graph_example in enumerate(scaled_graphs):
    if i % 10 == 0:
        print(f"进度: {i+1}/{len(scaled_graphs)} ({100*(i+1)/len(scaled_graphs):.1f}%)")
    
    all_names = [s[0] for s in graph_example.nodes(data=True)]
    attacker_names = [s[0] for s in graph_example.nodes(data=True) if (s[1]['features'][7] == 1) & (s[0] != 'ball')]
    defender_names = [s[0] for s in graph_example.nodes(data=True) if (s[1]['features'][7] == 0) & (s[0] != 'ball')]
    
    for defender in defender_names:
        results, performance = calculate_defender_influence(graph_example, loaded_model, defender, attacker_names, all_names, loaded_scaler, xT_grid)
        
        defender_df = pd.DataFrame({
            'graph_index': i,
            'Attacker': attacker_names,
            'Defender': defender,
            'match_id': int(GAME_ID),
            'Def_Distance': results['distances'],
            'Def_Attention': results['attentions'],
            'Def_Influence': results['influences'],
            'Def_Rel_Influence (%)': results['rel_influences'],
            'Att_Threat': results['threats'],
            'Total_Perf': performance
        })
        all_defender_dataframes.append(defender_df)

combined_df = pd.concat(all_defender_dataframes, ignore_index=True)
print(f"\n✓ 分析完成: {len(combined_df)} 条记录")

开始批量分析 43 个图...

进度: 1/43 (2.3%)
进度: 11/43 (25.6%)
进度: 21/43 (48.8%)
进度: 31/43 (72.1%)
进度: 41/43 (95.3%)

✓ 分析完成: 5203 条记录


## 8. 保存结果

In [8]:
output_file = f'{OUTPUT_DIR}/{GAME_ID}_defender_performance.csv'
combined_df.to_csv(output_file, index=False)
print(f"✓ 结果已保存到: {output_file}")
print(f"\n数据统计:")
print(f"  总记录数: {len(combined_df)}")
print(f"  唯一防守球员: {combined_df['Defender'].nunique()}")
print(f"  唯一进攻球员: {combined_df['Attacker'].nunique()}")

✓ 结果已保存到: results/player_eval_10517/10517_defender_performance.csv

数据统计:
  总记录数: 5203
  唯一防守球员: 22
  唯一进攻球员: 22


## 9. 结果分析

In [9]:
print("防守球员总体表现排名（前10）:")
defender_performance = combined_df.groupby('Defender')['Total_Perf'].mean().sort_values(ascending=False)
print(defender_performance.head(10))

print("\n平均影响力最大的防守球员（前10）:")
defender_influence = combined_df.groupby('Defender')['Def_Influence'].mean().sort_values()
print(defender_influence.head(10))

print("\n平均注意力权重最高的防守球员（前10）:")
defender_attention = combined_df.groupby('Defender')['Def_Attention'].mean().sort_values(ascending=False)
print(defender_attention.head(10))

防守球员总体表现排名（前10）:
Defender
Emiliano Martínez      2.730669
Nicolás Otamendi       2.648188
Ángel Di María         2.598256
Cristian Romero        2.591391
Alexis Mac Allister    2.579930
Enzo Fernandez         2.577506
Nicolas Tagliafico     2.555664
Julian Alvarez         2.508420
Rodrigo de Paul        2.503514
Nahuel Molina          2.487209
Name: Total_Perf, dtype: float64

平均影响力最大的防守球员（前10）:
Defender
Ousmane Dembele        0.003855
Jules Kounde           0.003929
Antoine Griezmann      0.003947
Olivier Giroud         0.003961
Aurélien Tchouaméni    0.004009
Raphael Varane         0.004058
Adrien Rabiot          0.004071
Kylian Mbappé          0.004074
Dayot Upamecano        0.004108
Theo Hernandez         0.004109
Name: Def_Influence, dtype: float64

平均注意力权重最高的防守球员（前10）:
Defender
Hugo Lloris           0.045508
Theo Hernandez        0.045064
Kylian Mbappé         0.044925
Emiliano Martínez     0.044907
Ángel Di María        0.044715
Jules Kounde          0.044700
Dayot Upamecano    

## 总结

本notebook完成了以下测试：

✓ 数据加载（2022世界杯决赛）
✓ 图创建和缩放
✓ 模型加载
✓ 防守球员影响力计算
✓ 批量分析
✓ 结果保存和统计

### 下一步：

- 使用 `PlayerEval_2022WC.ipynb` 分析其他比赛
- 进行更深入的防守球员移除实验
- 可视化分析结果

### 相关文件：

- `PlayerEval_2022WC.ipynb` - 通用球员评估工具
- `Visualisation_2022WC.ipynb` - 可视化工具
- `player_eval.py` - 原始Python脚本